# Group-wise Transformations with `.transform()`

When you use a standard `.groupby()` aggregation (like `.groupby('Category')['Sales'].mean()`), Pandas collapses your dataset. If you start with 1,000 rows and 5 unique categories, the output is collapsed down to just 5 rows.

But what if you want to compare each individual's value against their group's average? For example, you want a column that shows the group's average next to every individual row so you can subtract them.

The **`.transform()`** method solves this. Instead of collapsing the data, `.transform()` applies the aggregation function, calculates the group-level statistics, and then **broadcasts (replicates)** those values back to match the original DataFrame's size and index. This allows you to perform row-by-row comparisons against group averages instantly.

### Real-World Analogy
Imagine a classroom of students taking an exam.
*   **Standard `.groupby().mean()`**: The teacher calculates the average score for each grade level. The output is just a short list of averages (e.g., Grade 10: 82%, Grade 11: 78%).
*   **`.groupby().transform('mean')`**: The teacher writes the grade-level average on *every single student's individual report card* right next to their actual score. Every student can immediately see how they performed relative to their specific grade average.

### Code Examples

Let's create a DataFrame representing employees, their departments, and their monthly salaries:

In [1]:
import pandas as pd
import numpy as np

data = {
    'Name': ['Alice', 'Bob', 'Charlie', 'David', 'Eva', 'Frank'],
    'Department': ['HR', 'IT', 'IT', 'HR', 'IT', 'HR'],
    'Salary': [5000, 8000, 11000, 6000, 9500, 5500]
}

df = pd.DataFrame(data)
print("--- Original DataFrame ---")
print(df)

--- Original DataFrame ---
      Name Department  Salary
0    Alice         HR    5000
1      Bob         IT    8000
2  Charlie         IT   11000
3    David         HR    6000
4      Eva         IT    9500
5    Frank         HR    5500


#### Standard Aggregation vs. Transform
Watch what happens when we use `.mean()` versus `.transform('mean')`:

In [2]:
# Standard Groupby mean (collapses the rows)
collapsed = df.groupby('Department')['Salary'].mean()
print("--- Standard Groupby Mean ---")
print(collapsed)

# Transform mean (retains original shape and broadcasts)
broadcasted = df.groupby('Department')['Salary'].transform('mean')
print("--- Transform Mean ---")
print(broadcasted)

--- Standard Groupby Mean ---
Department
HR    5500.0
IT    9500.0
Name: Salary, dtype: float64
--- Transform Mean ---
0    5500.0
1    9500.0
2    9500.0
3    5500.0
4    9500.0
5    5500.0
Name: Salary, dtype: float64


#### B) Creating a Comparative Column
Because `.transform()` returns a Series with the exact same size and index as the original DataFrame, we can directly assign it as a new column:

In [3]:
# Add the department average salary next to each employee
df['Dept_Avg_Salary'] = df.groupby('Department')['Salary'].transform('mean')

# Calculate how much above or below the department average each employee is
df['Salary_Diff_From_Avg'] = df['Salary'] - df['Dept_Avg_Salary']
print(df)

      Name Department  Salary  Dept_Avg_Salary  Salary_Diff_From_Avg
0    Alice         HR    5000           5500.0                -500.0
1      Bob         IT    8000           9500.0               -1500.0
2  Charlie         IT   11000           9500.0                1500.0
3    David         HR    6000           5500.0                 500.0
4      Eva         IT    9500           9500.0                   0.0
5    Frank         HR    5500           5500.0                   0.0


### Common Pitfalls to Avoid
1.  **Passing Functions that Change Shape**: The function passed to `.transform()` must return a single aggregated value (like `'mean'`, `'sum'`, or `'std'`) or a Series of the same length. Passing a function that returns a collapsed structure will raise a `ValueError`.
2.  **Using `.transform()` on Non-Numerical Columns**: Trying to calculate a transform mean on a column containing string values will fail because mathematical operations cannot be applied to text.


#### Exercise 1 (Medium)
Given a dataset of retail store transactions:
```python
import pandas as pd
sales_data = pd.DataFrame({
    'Store': ['Store_A', 'Store_A', 'Store_B', 'Store_B', 'Store_A', 'Store_B'],
    'Product': ['Apple', 'Banana', 'Apple', 'Orange', 'Banana', 'Orange'],
    'Revenue': [100, 150, 200, 300, 120, 250]
})
```
Use `.transform()` to add a column called `Store_Total_Revenue` that shows the sum of revenue for each transaction's respective store. Then, calculate each transaction's percentage contribution to its store's total revenue.


In [5]:
import pandas as pd

sales_data = pd.DataFrame({
    'Store': ['Store_A', 'Store_A', 'Store_B', 'Store_B', 'Store_A', 'Store_B'],
    'Product': ['Apple', 'Banana', 'Apple', 'Orange', 'Banana', 'Orange'],
    'Revenue': [100, 150, 200, 300, 120, 250]
})

# Calculate store total and broadcast it
sales_data['Store_Total_Revenue'] = sales_data.groupby('Store')['Revenue'].transform('sum')

# Calculate percentage contribution
sales_data['Revenue_Contribution_Pct'] = (sales_data['Revenue'] / sales_data['Store_Total_Revenue']) * 100
print(sales_data)

     Store Product  Revenue  Store_Total_Revenue  Revenue_Contribution_Pct
0  Store_A   Apple      100                  370                 27.027027
1  Store_A  Banana      150                  370                 40.540541
2  Store_B   Apple      200                  750                 26.666667
3  Store_B  Orange      300                  750                 40.000000
4  Store_A  Banana      120                  370                 32.432432
5  Store_B  Orange      250                  750                 33.333333
